# Ejercicio: Web Scraping

## Objetivo de la práctica

El objetivo de este ejercicio es construir un web scraper que recoja datos de un website.

### Parte 0: Planificar
1. Identificar los datos que quieres obtener.
2. Elegir el sitio web objetivo.
3. Planificar la estructura del corpus.

## Parte 1: Entender el sitio web objetivo

- Analizar la estructura de la página web a ser analizada.
- Identificar los elementos HTML que contienen los datos bsuscados.

In [1]:
from bs4 import BeautifulSoup

file = './rotisserie-chicken.html'

# Load the HTML file
with open(file, "r", encoding="utf-8") as file:
    html_content = file.read()
    
# Parse the HTML content with BeautifulSoup
soup = BeautifulSoup(html_content, "html.parser")

In [2]:
# Extracting the recipe title
title = soup.find("meta", {"property": "og:title"})["content"]
title

'Rotisserie Chicken'

In [3]:
ingredients_section = soup.find_all("li", class_="mm-recipes-structured-ingredients__list-item")
for ingredient in ingredients_section:
    print(ingredient.text.strip())

1 (3 pound) whole chicken
1 pinch salt
¼ cup butter, melted
1 tablespoon salt
1 tablespoon ground paprika
¼ tablespoon ground black pepper


## Parte 2: Obtener los datos deseados

* Buscar dentro del contenido HTML y extraer la información.

In [4]:
# Extracting the description
description = soup.find("meta", {"name": "description"})["content"]

# Extracting the ingredients
ingredients_section = soup.find_all("li", class_="mm-recipes-structured-ingredients__list-item")
ingredients = [ingredient.get_text().strip() for ingredient in ingredients_section]

# Extracting the instructions
instructions_section = soup.find_all("p", class_="comp mntl-sc-block mntl-sc-block-html")
instructions = [instruction.get_text().strip() for instruction in instructions_section]

# Extracting the nutrition information
nutrition_section = soup.find_all("span", class_="mm-recipes-nutrition-facts-label__nutrient-name mm-recipes-nutrition-facts-label__nutrient-name--has-postfix")
nutrition_facts = [fact.parent.get_text().strip().replace('\n', ' ') for fact in nutrition_section]

# Print the extracted information
print("Recipe Title:", title)
print("Description:", description)
print("Ingredients:")
for ingredient in ingredients:
    print("-", ingredient)
print("Instructions:")
for i, instruction in enumerate(instructions, 1):
    print(f"{i}. {instruction}")
print("Nutrition Facts:")
for fact in nutrition_facts:
    print("-", fact)


Recipe Title: Rotisserie Chicken
Description: Rotisserie chicken that's easy to cook on a gas grill and turns out moist and juicy with crispy skin. This is a simple recipe that our family loves.
Ingredients:
- 1 (3 pound) whole chicken
- 1 pinch salt
- ¼ cup butter, melted
- 1 tablespoon salt
- 1 tablespoon ground paprika
- ¼ tablespoon ground black pepper
Instructions:
1. Intimidated by the idea of making a rotisserie chicken at home? We're here to help. Get your grill and rotisserie attachment ready — you'll want to try this recipe ASAP.
2. Here's what you'll need to make rotisserie chicken at home:
3. · Whole Chicken: This recipe is meant for a whole 3-pound chicken. If your chicken is larger or smaller, you'll have to adjust the cooking time.· Butter: Butter keeps the chicken moist and juicy, while giving the seasonings something to stick to.· Seasonings: The rotisserie chicken is simply seasoned with salt, pepper, and paprika.
4. You'll find the full, step-by-step recipe below — b

In [5]:
# Find all links to other recipes using data-tracking-target-url
tags = soup.find_all(attrs={"data-tracking-target-url": True})

recipe_urls = list({
    tag["data-tracking-target-url"]
    for tag in tags
    if "/recipe/" in tag["data-tracking-target-url"]
})
print("Linked Recipes:")
for url in recipe_urls:
    print(url)

Linked Recipes:
https://www.allrecipes.com/recipe/34957/easy-barbeque-chicken/
https://www.allrecipes.com/recipe/275044/grilled-chicken-under-a-brick/
https://www.allrecipes.com/recipe/258659/rosemary-buttermilk-chicken/
https://www.allrecipes.com/recipe/93168/rotisserie-chicken/
https://www.allrecipes.com/recipe/238575/cilantro-lime-grilled-chicken/
https://www.allrecipes.com/recipe/214619/bbq-beer-can-chicken/
https://www.allrecipes.com/recipe/19944/drunk-chicken/
https://www.allrecipes.com/recipe/274724/grilled-spatchcocked-chicken/
https://www.allrecipes.com/recipe/275062/buttermilk-barbecue-chicken/
https://www.allrecipes.com/recipe/264278/miso-honey-chicken/
https://www.allrecipes.com/recipe/281255/smoked-whole-chicken/
https://www.allrecipes.com/recipe/8998/darn-good-chicken/
https://www.allrecipes.com/recipe/214618/beer-can-chicken/
https://www.allrecipes.com/recipe/228070/the-best-beer-can-chicken-ever/
https://www.allrecipes.com/recipe/222936/smoked-beer-butt-chicken/
https:/

## Parte 3: Construir el corpus

* Descargar las páginas HTML de las recetas enlazadas y extraer su contenido usando el mismo scraper.

In [6]:
import requests
import time
import os
import json
from bs4 import BeautifulSoup

HEADERS = {
    "User-Agent": "Mozilla/5.0 (X11; Linux x86_64) AppleWebKit/537.36 (KHTML, like Gecko) Chrome/125.0.0.0 Safari/537.36",
    "Accept-Language": "en-US,en;q=0.9",
}

# Crear la carpeta donde se guardarán los archivos
CARPETA_RECETAS = "recetas_json"
os.makedirs(CARPETA_RECETAS, exist_ok=True)

def scrape_recipe(url: str) -> dict | None:
    try:
        r = requests.get(url, headers=HEADERS, timeout=15)
        r.raise_for_status()
    except Exception as e:
        print(f"  Error al descargar: {e}")
        return None

    soup = BeautifulSoup(r.text, "html.parser")

    # Extraer todos los datos
    title_tag = soup.find("meta", {"property": "og:title"})
    title = title_tag["content"] if title_tag else "Sin título"

    desc_tag = soup.find("meta", {"name": "description"})
    description = desc_tag["content"] if desc_tag else "Sin descripción"

    ingredients_section = soup.find_all("li", class_="mm-recipes-structured-ingredients__list-item")
    ingredients = [ingredient.get_text().strip() for ingredient in ingredients_section]

    instructions_section = soup.find_all("p", class_="comp mntl-sc-block mntl-sc-block-html")
    instructions = [instruction.get_text().strip() for instruction in instructions_section]

    nutrition_section = soup.find_all("span", class_="mm-recipes-nutrition-facts-label__nutrient-name mm-recipes-nutrition-facts-label__nutrient-name--has-postfix")
    nutrition_facts = [fact.parent.get_text().strip().replace('\n', ' ') for fact in nutrition_section]

    return {
        "url": url,
        "title": title,
        "description": description,
        "ingredients": ingredients,
        "instructions": instructions,
        "nutrition_facts": nutrition_facts
    }

for url in recipe_urls:
    print(f"\nProcesando: {url}")
    recipe = scrape_recipe(url)

    if recipe:
        print(f"  ✓ {recipe['title']}")

        # Guardar cada receta individualmente DENTRO DE LA CARPETA
        nombre_archivo = recipe['title'].replace(" ", "_").lower()[:50] + ".json"
        ruta_completa = os.path.join(CARPETA_RECETAS, nombre_archivo)

        with open(ruta_completa, 'w', encoding='utf-8') as f:
            json.dump(recipe, f, ensure_ascii=False, indent=2)

    time.sleep(1.5)

print(f"\n Recetas extraídas y guardadas exitosamente en la carpeta '{CARPETA_RECETAS}'")


Procesando: https://www.allrecipes.com/recipe/34957/easy-barbeque-chicken/
  ✓ Easy Barbeque Chicken

Procesando: https://www.allrecipes.com/recipe/275044/grilled-chicken-under-a-brick/
  ✓ Grilled Chicken Under a Brick

Procesando: https://www.allrecipes.com/recipe/258659/rosemary-buttermilk-chicken/
  ✓ Rosemary Buttermilk Chicken

Procesando: https://www.allrecipes.com/recipe/93168/rotisserie-chicken/
  ✓ Rotisserie Chicken

Procesando: https://www.allrecipes.com/recipe/238575/cilantro-lime-grilled-chicken/
  ✓ Cilantro-Lime Grilled Chicken

Procesando: https://www.allrecipes.com/recipe/214619/bbq-beer-can-chicken/
  ✓ Best Beer Can Chicken

Procesando: https://www.allrecipes.com/recipe/19944/drunk-chicken/
  ✓ Drunk Chicken

Procesando: https://www.allrecipes.com/recipe/274724/grilled-spatchcocked-chicken/
  ✓ Grilled Spatchcocked Chicken

Procesando: https://www.allrecipes.com/recipe/275062/buttermilk-barbecue-chicken/
  ✓ Buttermilk Barbecue Chicken

Procesando: https://www.allr

## Parte 4: Hacer RAG con las recetas obtenidas
* Una vez que se ha construido el corpus, implementar y desplegar RAG para realizar búsquedas en el corpus

In [7]:
import chromadb
from sentence_transformers import SentenceTransformer

embed_model = SentenceTransformer("intfloat/e5-base-v2")

def embed_passages(texts: list[str]) -> list[list[float]]:
    return embed_model.encode(
        ["passage: " + t for t in texts],
        normalize_embeddings=True,
    ).tolist()

def embed_query_rag(query: str) -> list[float]:
    return embed_model.encode(
        ["query: " + query],
        normalize_embeddings=True,
    )[0].tolist()

/home/migueldev/Documents/University/Courses/RI/Practices/recuperacion_de_informacion/.venv/lib/python3.12/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm
Loading weights: 100%|██████████| 199/199 [00:00<00:00, 2969.40it/s]


In [10]:
chroma_client = chromadb.PersistentClient(path="./chroma_recetas_db")

try:
    chroma_client.delete_collection("recetas_cocina")
    print("Colección anterior eliminada")
except Exception:
    pass

collection = chroma_client.get_or_create_collection(name="recetas_cocina")

CARPETA_RECETAS = "recetas_json"

ids, documents, metadatas = [], [], []
for i, filename in enumerate(sorted(os.listdir(CARPETA_RECETAS))):
    if filename.endswith(".json"):
        with open(os.path.join(CARPETA_RECETAS, filename), "r", encoding="utf-8") as f:
            recipe = json.load(f)
        doc_text = (
            f"Título: {recipe['title']}\n"
            f"Descripción: {recipe['description']}\n"
            f"Ingredientes: {', '.join(recipe['ingredients'])}\n"
            f"Instrucciones: {' '.join(recipe['instructions'])}"
        )
        ids.append(str(i))
        documents.append(doc_text)
        metadatas.append({"title": recipe["title"], "url": recipe.get("url", "")})

collection.add(
    ids=ids,
    documents=documents,
    embeddings=embed_passages(documents),
    metadatas=metadatas,
)
print(f"✓ {len(ids)} recetas indexadas con e5-base-v2")

Colección anterior eliminada
✓ 17 recetas indexadas con e5-base-v2


In [13]:
from google import genai

GEMINI_API_KEY ="AQ.xxxxxxxxxxx"
gemini = genai.Client(api_key=GEMINI_API_KEY)

def rag_query(question: str, k: int = 3) -> str:
    results = collection.query(
        query_embeddings=[embed_query_rag(question)],
        n_results=k,
        include=["documents", "metadatas"],
    )

    print("Documentos recuperados:")
    for i, (doc, meta) in enumerate(zip(results["documents"][0], results["metadatas"][0]), 1):
        print(f"  {i}. [{meta['title']}] {meta['url']}")

    context = "\n\n---\n\n".join(
        f"[{m['title']}]\n{d}"
        for d, m in zip(results["documents"][0], results["metadatas"][0])
    )
    prompt = (
        "Eres un asistente experto en recetas de cocina.\n"
        "Responde usando SOLO el contexto proporcionado. Si no puedes, dilo.\n\n"
        f"Contexto:\n{context}\n\n"
        f"Pregunta: {question}\n\nRespuesta:"
    )
    response = gemini.models.generate_content(model="gemini-2.5-flash", contents=prompt)
    return response.text

questions = [
    "¿Qué receta lleva paprika?",
    "¿Cuánto tiempo necesita el pollo en la parrilla?",
    "¿Qué receta usa mantequilla y limón?",
]

for q in questions:
    print(f"\n{'='*60}")
    print(f"Pregunta: {q}")
    print(f"{'='*60}")
    answer = rag_query(q)
    print(f"\nRespuesta Gemini:\n{answer}")


Pregunta: ¿Qué receta lleva paprika?
Documentos recuperados:
  1. [Good Frickin’ Paprika Chicken] https://www.allrecipes.com/recipe/221093/good-frickin-paprika-chicken/
  2. [Smoked Whole Chicken] https://www.allrecipes.com/recipe/281255/smoked-whole-chicken/
  3. [Rosemary Buttermilk Chicken] https://www.allrecipes.com/recipe/258659/rosemary-buttermilk-chicken/

Respuesta Gemini:
Las siguientes recetas llevan paprika:

*   Good Frickin’ Paprika Chicken
*   Smoked Whole Chicken
*   Rosemary Buttermilk Chicken

Pregunta: ¿Cuánto tiempo necesita el pollo en la parrilla?
Documentos recuperados:
  1. [Good Frickin’ Paprika Chicken] https://www.allrecipes.com/recipe/221093/good-frickin-paprika-chicken/
  2. [Smoked Whole Chicken] https://www.allrecipes.com/recipe/281255/smoked-whole-chicken/
  3. [Beer Butt Chicken] https://www.allrecipes.com/recipe/14531/beer-butt-chicken/

Respuesta Gemini:
Según el contexto proporcionado:

*   Para la receta "Good Frickin’ Paprika Chicken", el pollo se 